# Deploy da Plataforma JaguarData na AWS

Notebook executável de dentro de um **SageMaker Studio** da conta AWS alvo.
Provisiona, via Terraform, a plataforma inteira:

- **Backend** FastAPI em ECS Fargate atrás de um ALB
- **Banco** SQLite em volume EFS persistente
- **Frontend** React em S3 + CloudFront
- CloudFront único com 2 origens: `default → S3`, `/api/* → ALB` (sem CORS)

Rode as células em ordem, de cima para baixo.

## Pré-requisitos

1. **Execution role do SageMaker** com permissão para criar a infra: na prática
   `PowerUserAccess` + `IAMFullAccess` (cria VPC, ECS, ALB, EFS, IAM, S3,
   CloudFront, Secrets Manager, ECR, DynamoDB).
2. **Build da imagem** usa `sagemaker-studio-image-build`, que delega ao
   **AWS CodeBuild** (o Studio clássico não tem daemon Docker). O role precisa
   de permissões de CodeBuild + ECR, e do trust para `codebuild.amazonaws.com`.
3. **Bedrock habilitado** na região escolhida (o backend chama Bedrock).
4. O repositório `Plataforma-Agentica` clonado no Studio; este notebook está em
   `infra/platform/` dentro dele.

> O `JWT_SECRET` é gerado aleatoriamente a cada execução e gravado no Secrets
> Manager. Reexecutar o notebook rotaciona o segredo.

In [ ]:
# --- Configuração -----------------------------------------------------------
import boto3, secrets, pathlib, subprocess, sys, os

REGION  = "us-east-1"          # ajuste se necessário (precisa ter Bedrock)
PROJECT = "jaguardata"
IMAGE_TAG = "latest"

ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

# Notebook está em infra/platform/ — a raiz do repo é dois níveis acima.
REPO_ROOT    = pathlib.Path.cwd().parents[1]
TF_DIR       = REPO_ROOT / "infra" / "platform"
BACKEND_DIR  = REPO_ROOT / "backend"
FRONTEND_DIR = REPO_ROOT / "frontend"

STATE_BUCKET = f"{PROJECT}-tfstate-{ACCOUNT_ID}"
LOCK_TABLE   = f"{PROJECT}-tf-lock"
JWT_SECRET   = secrets.token_urlsafe(48)

def run(cmd, cwd=None):
    """Executa um comando de shell, faz stream da saída e falha em returncode != 0."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"comando falhou (exit {r.returncode}): {cmd}")
    return r.stdout

print("account:", ACCOUNT_ID, "| region:", REGION)
print("repo root:", REPO_ROOT)
assert TF_DIR.exists(), f"infra/platform não encontrado em {TF_DIR}"

In [ ]:
# --- Instalar o binário do Terraform ---------------------------------------
import urllib.request, zipfile

TF_VERSION = "1.9.8"
bindir = pathlib.Path.home() / "bin"
bindir.mkdir(exist_ok=True)

if not (bindir / "terraform").exists():
    url = (f"https://releases.hashicorp.com/terraform/{TF_VERSION}/"
           f"terraform_{TF_VERSION}_linux_amd64.zip")
    urllib.request.urlretrieve(url, "/tmp/terraform.zip")
    with zipfile.ZipFile("/tmp/terraform.zip") as z:
        z.extractall(bindir)
    os.chmod(bindir / "terraform", 0o755)

os.environ["PATH"] = f"{bindir}:{os.environ['PATH']}"
run("terraform version")

In [ ]:
# --- Instalar tooling de build ---------------------------------------------
# sagemaker-studio-image-build: builda a imagem do backend via CodeBuild.
run(f"{sys.executable} -m pip install --quiet sagemaker-studio-image-build")

# Node/npm para o build do frontend (o kernel Python não traz npm).
try:
    run("node --version && npm --version")
except RuntimeError:
    run("conda install -y -q nodejs")
    run("node --version && npm --version")

In [ ]:
# --- Bootstrap do state remoto do Terraform (S3 + DynamoDB) ----------------
s3  = boto3.client("s3", region_name=REGION)
ddb = boto3.client("dynamodb", region_name=REGION)

try:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=STATE_BUCKET)
    else:
        s3.create_bucket(Bucket=STATE_BUCKET,
                         CreateBucketConfiguration={"LocationConstraint": REGION})
    print("bucket de state criado:", STATE_BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    print("bucket de state já existe:", STATE_BUCKET)

s3.put_bucket_versioning(Bucket=STATE_BUCKET,
                         VersioningConfiguration={"Status": "Enabled"})

try:
    ddb.create_table(TableName=LOCK_TABLE,
                     AttributeDefinitions=[{"AttributeName": "LockID", "AttributeType": "S"}],
                     KeySchema=[{"AttributeName": "LockID", "KeyType": "HASH"}],
                     BillingMode="PAY_PER_REQUEST")
    ddb.get_waiter("table_exists").wait(TableName=LOCK_TABLE)
    print("tabela de lock criada:", LOCK_TABLE)
except ddb.exceptions.ResourceInUseException:
    print("tabela de lock já existe:", LOCK_TABLE)

In [ ]:
# --- terraform.tfvars + terraform init -------------------------------------
def write_tfvars(image_uri=""):
    (TF_DIR / "terraform.tfvars").write_text(
        f'aws_region        = "{REGION}"\n'
        f'project_name      = "{PROJECT}"\n'
        f'jwt_secret        = "{JWT_SECRET}"\n'
        f'desired_count     = 1\n'
        f'backend_image_uri = "{image_uri}"\n'
    )

write_tfvars()
run(f'terraform init -reconfigure '
    f'-backend-config="bucket={STATE_BUCKET}" '
    f'-backend-config="region={REGION}" '
    f'-backend-config="dynamodb_table={LOCK_TABLE}"', cwd=TF_DIR)
run("terraform validate", cwd=TF_DIR)

In [ ]:
# --- Fase A: criar só o ECR (a imagem precisa de um destino antes do build) -
run("terraform apply -auto-approve "
    "-target=aws_ecr_repository.backend "
    "-target=aws_ecr_lifecycle_policy.backend", cwd=TF_DIR)

ECR_URL = run("terraform output -raw ecr_repository_url", cwd=TF_DIR).strip()
print("ECR repo:", ECR_URL)

In [ ]:
# --- Build da imagem do backend (CodeBuild via sm-docker) -------------------
# O contexto é backend/ — o Dockerfile copia app/ e pyproject.toml.
run(f"sm-docker build . --repository {PROJECT}-backend:{IMAGE_TAG} "
    f"--region {REGION}", cwd=BACKEND_DIR)

IMAGE_URI = f"{ECR_URL}:{IMAGE_TAG}"
print("imagem do backend:", IMAGE_URI)

In [ ]:
# --- Build do frontend ------------------------------------------------------
# .env.production já define VITE_API_BASE=/api — build agnóstico de ambiente.
run("npm ci", cwd=FRONTEND_DIR)
run("npm run build", cwd=FRONTEND_DIR)
assert (FRONTEND_DIR / "dist" / "index.html").exists(), "build do frontend falhou"

In [ ]:
# --- Fase B: apply completo (VPC, EFS, ALB, ECS, S3, CloudFront) -----------
write_tfvars(image_uri=IMAGE_URI)
run("terraform apply -auto-approve", cwd=TF_DIR)

In [ ]:
# --- Coletar outputs --------------------------------------------------------
import json
out = json.loads(run("terraform output -json", cwd=TF_DIR))
BUCKET    = out["frontend_bucket"]["value"]
CF_DOMAIN = out["cloudfront_domain"]["value"]
CF_ID     = out["cloudfront_distribution_id"]["value"]
print("frontend bucket:", BUCKET)
print("cloudfront     :", CF_DOMAIN)

In [ ]:
# --- Publicar o frontend no S3 + invalidar o CloudFront --------------------
import time
run(f"aws s3 sync dist/ s3://{BUCKET}/ --delete --region {REGION}", cwd=FRONTEND_DIR)

cf = boto3.client("cloudfront")
cf.create_invalidation(
    DistributionId=CF_ID,
    InvalidationBatch={
        "Paths": {"Quantity": 1, "Items": ["/*"]},
        "CallerReference": str(time.time()),
    })
print("frontend publicado e invalidação criada")

In [ ]:
# --- Smoke test -------------------------------------------------------------
# /api/* é roteado para o ALB; /api/health responde quando o ECS está saudável.
import urllib.request, time

health = f"{CF_DOMAIN}/api/health"
for attempt in range(12):
    try:
        body = urllib.request.urlopen(health, timeout=10).read().decode()
        print("OK:", health, "->", body)
        break
    except Exception as e:
        print(f"[{attempt + 1}/12] aguardando ECS/CloudFront subir... ({e})")
        time.sleep(30)
else:
    print("AVISO: health check não respondeu. Veja os logs em CloudWatch "
          f"/ecs/{PROJECT}-backend e os eventos do ECS service.")

print("\nPlataforma:", CF_DOMAIN)

## Teardown

A célula abaixo **destrói toda a infra da plataforma**. Está comentada de
propósito — descomente e execute apenas quando quiser remover tudo.

O bucket de state do Terraform e a tabela de lock **não** são removidos por
`terraform destroy`; apague-os manualmente depois, se desejar.

In [ ]:
# CUIDADO: remove VPC, ECS, ALB, EFS (com os dados SQLite), S3 e CloudFront.
# run("terraform destroy -auto-approve", cwd=TF_DIR)